### How to Mount Google Drive in Colab

Mounting Google Drive allows your Colab notebook to access files and folders stored in your Google Drive account. This is particularly useful for:

*   **Persistent Storage**: Colab's default runtime storage is temporary. Mounting Drive provides a persistent location for your data.
*   **Accessing Datasets**: Easily load large datasets or models stored in your Drive.
*   **Saving Results**: Save your model outputs, figures, or processed data directly back to your Drive.

**Steps:**

1.  **Execute the Code Below**: Run the Python cell below. It will prompt you to authorize Colab to access your Google Drive.
2.  **Follow the Authorization Link**: A link will appear in the output. Click on it, sign in to your Google Account, and grant the necessary permissions.
3.  **Copy and Paste the Authorization Code**: After granting permissions, you'll receive an authorization code. Copy this code.
4.  **Paste into Colab**: Paste the authorization code into the input box that appears in the Colab output and press Enter.

Once successfully mounted, your Google Drive contents will be accessible under the `/content/drive/MyDrive/` path in the Colab file system.

In [29]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


After executing the cell above and successfully authorizing, you can explore your Drive files directly within the Colab environment. For example, if you have a folder named `my_data` in your Google Drive's root, you can access it via `/content/drive/MyDrive/my_data`.

### `sync_directories` Function Overview

This function will recursively traverse a source directory and copy its files into a destination directory. It includes logic to:

1.  **Check Directory Existence**: Verify if source and destination directories exist.
2.  **Handle Empty Source**: If the source directory is empty, it will issue a warning and stop, as there's nothing to sync.
3.  **Handle Empty Destination**: If the destination is empty, all files from the source will be copied as new.
4.  **Conditional Copying**: If a file exists in both source and destination, it will only be copied if the source file is newer than the destination file (based on modification times).

#### How to handle 'uploading directories' in Colab:

Since I cannot directly 'pop out a menu' for uploading, here's how you can manage directories and files in Google Colab:

*   **Manual Upload**: Click the 'Files' icon on the left sidebar (folder icon). You can then click the 'Upload' icon (file with an arrow) to upload individual files, or drag and drop files/folders directly into the file browser panel. For entire directory structures, you might need to upload a zipped archive and then unzip it using `!unzip your_archive.zip` in a code cell.
*   **Drive Mount**: For larger datasets or persistent storage, it's often best to mount your Google Drive: `from google.colab import drive; drive.mount('/content/drive')`.
*   **Git Clone**: If your data is in a Git repository, use `!git clone [repository_url]`.

In [30]:
import os
import shutil
import time
from tqdm.notebook import tqdm # Import tqdm for progress bars

def sync_directories(src_dir, dest_dir):
    """
    Copies files from source directory tree to destination directory tree,
    skipping files in destination that are newer than source.
    Handles directory existence and content checks with user guidance.

    Args:
        src_dir (str): The path to the source directory.
        dest_dir (str): The path to the destination directory.
    """

    # --- Check for source directory existence and content ---
    if not os.path.isdir(src_dir):
        print(f"Error: Source directory '{src_dir}' does not exist. Please check the 'Source Dir:' input field above and ensure the path is correct or upload the directory content.")
        return

    if not os.listdir(src_dir):
        print(f"Warning: Source directory '{src_dir}' is empty. Nothing to copy. Please upload content to this directory, or specify a non-empty source path in the 'Source Dir:' input field.")
        return

    # --- Check for destination directory existence and content ---
    if not os.path.isdir(dest_dir):
        print(f"Destination directory '{dest_dir}' does not exist. Creating it.")
        os.makedirs(dest_dir)
    elif not os.listdir(dest_dir):
        print(f"Warning: Destination directory '{dest_dir}' is empty. All source files will be copied as new. If you intended to select a pre-populated destination, please update the 'Dest Dir:' input field.")

    print(f"Starting synchronization from '{src_dir}' to '{dest_dir}'...")

    # --- Pre-computation phase to determine total files and bytes to copy ---
    files_to_copy_count = 0
    bytes_to_copy_count = 0
    files_skipped = 0
    bytes_skipped = 0

    for root, _, files in os.walk(src_dir):
        for file_name in files:
            source_file_path = os.path.join(root, file_name)
            relative_path = os.path.relpath(root, src_dir)
            current_dest_path = os.path.join(dest_dir, relative_path)
            destination_file_path = os.path.join(current_dest_path, file_name)

            should_copy = True
            if os.path.exists(destination_file_path):
                try:
                    src_mtime = os.path.getmtime(source_file_path)
                    dest_mtime = os.path.getmtime(destination_file_path)

                    if dest_mtime > src_mtime:
                        should_copy = False
                    elif dest_mtime == src_mtime:
                        should_copy = False
                except Exception:
                    # If mtime comparison fails, assume it needs copying
                    pass

            if should_copy:
                files_to_copy_count += 1
                try:
                    bytes_to_copy_count += os.path.getsize(source_file_path)
                except OSError: # Handle cases where file might be inaccessible or non-existent
                    pass # Will count it as a file but not add to bytes

    if files_to_copy_count == 0:
        print("No files need to be copied or updated.")
        return

    # --- Progress Bar Initialization ---
    file_pbar = tqdm(total=files_to_copy_count, unit='file', desc='Files Synced')
    byte_pbar = tqdm(total=bytes_to_copy_count, unit='B', unit_scale=True, desc='Bytes Synced', leave=True)

    # --- Actual Copying Phase ---
    for root, dirs, files in os.walk(src_dir):
        # Construct the relative path from src_dir to the current 'root'
        relative_path = os.path.relpath(root, src_dir)
        current_dest_path = os.path.join(dest_dir, relative_path)

        # Create corresponding directories in the destination
        if not os.path.exists(current_dest_path):
            os.makedirs(current_dest_path)
            # print(f"  Created directory: {current_dest_path}") # Suppressed for cleaner pbar output

        for file_name in files:
            source_file_path = os.path.join(root, file_name)
            destination_file_path = os.path.join(current_dest_path, file_name)

            should_copy = True
            # Re-evaluate should_copy based on actual destination state
            if os.path.exists(destination_file_path):
                try:
                    src_mtime = os.path.getmtime(source_file_path)
                    dest_mtime = os.path.getmtime(destination_file_path)

                    if dest_mtime > src_mtime:
                        should_copy = False
                    elif dest_mtime == src_mtime:
                        should_copy = False
                except Exception as e:
                    print(f"  Error comparing modification times for '{file_name}': {e}. Attempting to copy.")

            if should_copy:
                try:
                    shutil.copy2(source_file_path, destination_file_path)
                    file_pbar.update(1)
                    byte_pbar.update(os.path.getsize(source_file_path))
                    # print(f"  Copied '{file_name}' to '{destination_file_path}'") # Suppressed for cleaner pbar output
                except Exception as e:
                    print(f"  Error copying '{file_name}': {e}") # Keep error messages
            else:
                files_skipped += 1
                bytes_skipped += os.path.getsize(source_file_path)

    file_pbar.close()
    byte_pbar.close()
    print(f"Files skipped: {files_skipped}")
    print(f"Bytes skipped: {bytes_skipped}")
    print(f"Synchronization from '{src_dir}' to '{dest_dir}' complete.")

### Demonstration of the `sync_directories` function

This example creates a sample source directory with subdirectories and files, and a destination directory with some pre-existing files (some newer than source, some older). Then, it calls `sync_directories` and shows the outcome. Finally, it cleans up the created dummy directories.

In [31]:
# --- Setup: Create dummy source and destination directories and files ---
demo_src_root = 'demo_source_tree'
demo_dest_root = 'demo_destination_tree'

# Clean up any previous demo runs
if os.path.exists(demo_src_root): shutil.rmtree(demo_src_root)
if os.path.exists(demo_dest_root): shutil.rmtree(demo_dest_root)

# Create source structure
os.makedirs(os.path.join(demo_src_root, 'subdir1'), exist_ok=True)
os.makedirs(os.path.join(demo_src_root, 'subdir2'), exist_ok=True)

with open(os.path.join(demo_src_root, 'fileA.txt'), 'w') as f: f.write('Source A content')
with open(os.path.join(demo_src_root, 'subdir1', 'fileB.txt'), 'w') as f: f.write('Source B content')
with open(os.path.join(demo_src_root, 'subdir2', 'fileC.txt'), 'w') as f: f.write('Source C content')

# Create an 'older' file for update demonstration
with open(os.path.join(demo_src_root, 'fileD.txt'), 'w') as f: f.write('Source D content - newer')
# Set mtime to be older for fileD.txt so it gets copied
os.utime(os.path.join(demo_src_root, 'fileD.txt'), (time.time() - 1000, time.time() - 1000))

# Create an 'identical' file for skip demonstration
with open(os.path.join(demo_src_root, 'fileE.txt'), 'w') as f: f.write('Source E content - identical mtime')
# Set mtime to match what we'll create in dest for fileE.txt
os.utime(os.path.join(demo_src_root, 'fileE.txt'), (time.time() + 10, time.time() + 10))


print(f"Created dummy source directory: {demo_src_root}")

# Create destination structure and pre-populate with some files
os.makedirs(os.path.join(demo_dest_root, 'subdir1'), exist_ok=True)

# FileA.txt will be new in dest
# FileB.txt will be new in dest (in subdir1)
# FileC.txt will be new in dest (in subdir2, which needs to be created)

# Create fileD.txt in dest that is newer than src's fileD.txt - should be skipped
with open(os.path.join(demo_dest_root, 'fileD.txt'), 'w') as f: f.write('Destination D content - NEWER')
# Set mtime to be newer for fileD.txt in dest
os.utime(os.path.join(demo_dest_root, 'fileD.txt'), (time.time() + 1000, time.time() + 1000))

# Create fileE.txt in dest with identical mtime - should be skipped
with open(os.path.join(demo_dest_root, 'fileE.txt'), 'w') as f: f.write('Destination E content - IDENTICAL mtime')
# Set mtime to match src for fileE.txt
os.utime(os.path.join(demo_dest_root, 'fileE.txt'), (time.time() + 10, time.time() + 10))

# Create a file unique to destination - should remain untouched
with open(os.path.join(demo_dest_root, 'dest_only_file.txt'), 'w') as f: f.write('This file is only in destination')

print(f"Created dummy destination directory: {demo_dest_root}")

print("\n--- Running sync_directories ---")
sync_directories(demo_src_root, demo_dest_root)
print("------------------------------\n")

print(f"Files in '{demo_dest_root}' after synchronization:")
for root, dirs, files in os.walk(demo_dest_root):
    level = root.replace(demo_dest_root, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f'{{}}{{}}/'.format(indent, os.path.basename(root)))
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f'{{}}{{}}'.format(subindent, f))

#print("\n--- Cleanup ---")
#shutil.rmtree(demo_src_root)
#shutil.rmtree(demo_dest_root)
#print("Dummy directories and files cleaned up.")

Created dummy source directory: demo_source_tree
Created dummy destination directory: demo_destination_tree

--- Running sync_directories ---
Starting synchronization from 'demo_source_tree' to 'demo_destination_tree'...


Files Synced:   0%|          | 0/3 [00:00<?, ?file/s]

Bytes Synced:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

Files skipped: 2
Bytes skipped: 58
Synchronization from 'demo_source_tree' to 'demo_destination_tree' complete.
------------------------------

Files in 'demo_destination_tree' after synchronization:
demo_destination_tree/
    dest_only_file.txt
    fileE.txt
    fileA.txt
    fileD.txt
    subdir2/
        fileC.txt
    subdir1/
        fileB.txt


In [32]:
sync_directories(src_dir="", dest_dir="")

Error: Source directory '' does not exist. Please check the 'Source Dir:' input field above and ensure the path is correct or upload the directory content.


### 2. Directory Path Selection and `sync_directories` Execution using `ipywidgets.Text`

For selecting source and destination *directories* for functions like `sync_directories`, `ipywidgets.Text` is more suitable. You can type or paste the paths to the directories that exist (or will exist) in your Colab environment.

Remember, if these directories don't exist, the `sync_directories` function will guide you on how to proceed. If you need to upload entire directories, refer to the 'How to handle 'uploading directories' in Colab' section above.

In [33]:
# ex. source directory - "/content/drive/Othercomputers/My PC"
# ex. destination directory - "/content/drive/MyDrive/......"


# Input widgets for source and destination directories
import re # Import regex for path validation

src_dir_input = widgets.Text(
    value='demo_source_tree', # Pre-fill with example path from previous demo
    placeholder='Type source directory path',
    description='Source Dir:',
    disabled=False
)
dest_dir_input = widgets.Text(
    value='demo_destination_tree', # Pre-fill with example path from previous demo
    placeholder='Type destination directory path',
    description='Dest Dir:',
    disabled=False
)

# Button to trigger synchronization
sync_button = widgets.Button(
    description='Run Synchronization',
    disabled=False,
    button_style='primary', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Click to synchronize directories',
    icon='copy' # (FontAwesome names available here: https://fontawesome.com/icons?d=gallery&m=free)
)

# Output widget to display results
output_area = widgets.Output()

def _is_likely_windows_path(path):
    """
    Checks if a given path string likely represents a Windows-style path.
    """
    # Checks for drive letter (e.g., C:) or backslashes
    return bool(re.match(r'^[a-zA-Z]:\\', path)) or ('\\' in path and '/' not in path)


def on_sync_button_clicked(b):
    with output_area:
        output_area.clear_output()
        source = src_dir_input.value.strip()
        destination = dest_dir_input.value.strip()

        if not source or not destination:
            print("Please provide both source and destination directory paths.")
            return

        # Add a warning for Windows-style paths
        if _is_likely_windows_path(source):
            print(f"Warning: The source path '{source}' appears to be a Windows-style path. Colab runs on a Linux file system. Please ensure the path refers to a location accessible within Colab (e.g., '/content/my_folder' or a mounted Google Drive path).")
            print("You may need to upload your files or mount Google Drive first.")
            return # Stop execution if it's a suspicious path

        if _is_likely_windows_path(destination):
            print(f"Warning: The destination path '{destination}' appears to be a Windows-style path. Colab runs on a Linux file system. Please ensure the path refers to a location accessible within Colab (e.g., '/content/my_folder' or a mounted Google Drive path).")
            print("You may need to upload your files or mount Google Drive first.")
            return # Stop execution if it's a suspicious path

        print(f"Attempting to sync from '{source}' to '{destination}'")
        # Call the sync_directories function
        sync_directories(source, destination)
        print("Synchronization attempt completed. Check console output for details.")

sync_button.on_click(on_sync_button_clicked)

print("Enter source and destination directory paths and click 'Run Synchronization':")
display(src_dir_input, dest_dir_input, sync_button, output_area)

Enter source and destination directory paths and click 'Run Synchronization':


Text(value='demo_source_tree', description='Source Dir:', placeholder='Type source directory path')

Text(value='demo_destination_tree', description='Dest Dir:', placeholder='Type destination directory path')

Button(button_style='primary', description='Run Synchronization', icon='copy', style=ButtonStyle(), tooltip='C…

Output()